# Weather
- Source: MeteoSwiss
- https://opendatadocs.meteoswiss.ch/a-data-groundbased/a1-automatic-weather-stations

In [1]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

EXT_DIR  = "../data/external"

## metadata

In [ ]:
# stations = pd.read_csv(f"{EXT_DIR}/weather/ogd-smn_meta_stations.csv", encoding="latin-1", sep=";")
# parameters = pd.read_csv(f"{EXT_DIR}/weather/ogd-smn_meta_parameters.csv", encoding="latin-1", sep=";")
# data_inventory = pd.read_csv(f"{EXT_DIR}/weather/ogd-smn_meta_datainventory.csv", encoding="latin-1", sep=";")

stations = pd.read_csv("https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/ogd-smn_meta_stations.csv", encoding="latin-1", sep=";")
parameters = pd.read_csv("https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/ogd-smn_meta_parameters.csv", encoding="latin-1", sep=";")
data_inventory = pd.read_csv("https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/ogd-smn_meta_datainventory.csv", encoding="latin-1", sep=";")

In [29]:
stations[stations['station_canton'] == "ZH"]

,station_abbr,station_name,station_canton,station_wigos_id,station_type_de,station_type_fr,station_type_it,station_type_en,station_dataowner,station_data_since,station_height_masl,station_height_barometer_masl,station_coordinates_lv95_east,station_coordinates_lv95_north,station_coordinates_wgs84_lat,station_coordinates_wgs84_lon,station_exposition_de,station_exposition_fr,station_exposition_it,station_exposition_en,station_url_de,station_url_fr,station_url_it,station_url_en
76,HOE,Hörnli,ZH,0-20000-0-06689,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,01.08.1974,1133.0,1134.0,2713517.0,1247756.0,47.370864,8.941644,Gipfellage,Sommet,Vetta,crest,https://www.meteoschweiz.admin.ch/service-und-...,https://www.meteosuisse.admin.ch/services-et-p...,https://www.meteosvizzera.admin.ch/servizi-e-p...,https://www.meteoswiss.admin.ch/services-and-p...
80,KLO,Zürich / Kloten,ZH,0-20000-0-06670,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,01.01.1935,426.0,428.0,2682711.0,1259339.0,47.479611,8.535961,Ebene,Plaine,Pianura,plain,https://www.meteoschweiz.admin.ch/service-und-...,https://www.meteosuisse.admin.ch/services-et-p...,https://www.meteosvizzera.admin.ch/servizi-e-p...,https://www.meteoswiss.admin.ch/services-and-p...
83,LAE,Lägern,ZH,0-20000-0-06669,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,21.11.1989,845.0,NaN,2672251.0,1259460.0,47.481933,8.397222,Gipfellage,Sommet,Vetta,crest,https://www.meteoschweiz.admin.ch/service-und-...,https://www.meteosuisse.admin.ch/services-et-p...,https://www.meteosvizzera.admin.ch/servizi-e-p...,https://www.meteoswiss.admin.ch/services-and-p...
109,PFA,"Pfäffikon, ZH",ZH,0-756-0-PFA,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,01.01.1901,537.0,538.0,2699401.0,1248165.0,47.376817,8.754864,Ebene,Plaine,Pianura,plain,https://www.meteoschweiz.admin.ch/service-und-...,https://www.meteosuisse.admin.ch/services-et-p...,https://www.meteosvizzera.admin.ch/servizi-e-p...,https://www.meteoswiss.admin.ch/services-and-p...
119,REH,Zürich / Affoltern,ZH,0-20000-0-06664,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,01.01.1961,444.0,445.0,2681433.0,1253548.0,47.427694,8.517953,Ebene,Plaine,Pianura,plain,https://www.meteoschweiz.admin.ch/service-und-...,https://www.meteosuisse.admin.ch/services-et-p...,https://www.meteosvizzera.admin.ch/servizi-e-p...,https://www.meteoswiss.admin.ch/services-and-p...
134,SMA,Zürich / Fluntern,ZH,0-20000-0-06660,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,01.01.1864,556.0,557.0,2685118.0,1248066.0,47.377925,8.565742,Südhang,Versant sud,Versante meridionale,southern oriented slope,https://www.meteoschweiz.admin.ch/service-und-...,https://www.meteosuisse.admin.ch/services-et-p...,https://www.meteosvizzera.admin.ch/servizi-e-p...,https://www.meteoswiss.admin.ch/services-and-p...
144,UEB,Uetliberg,ZH,0-20000-0-06677,Automatische Wetterstationen - Messwerte,Stations météorologiques automatiques - Valeur...,Stazioni meteorologiche automatiche - Valori m...,Automatic weather stations - Measurement values,MeteoSchweiz,14.11.1991,854.0,NaN,2679455.0,1245034.0,47.351364,8.4

In [30]:
parameters[parameters['parameter_granularity'] == "H"]

,parameter_shortname,parameter_description_de,parameter_description_fr,parameter_description_it,parameter_description_en,parameter_group_de,parameter_group_fr,parameter_group_it,parameter_group_en,parameter_granularity,parameter_decimals,parameter_datatype,parameter_unit
1,dkl010h0,Windrichtung; Stundenmittel,Direction du vent; moyenne horaire,Direzione del vento; media oraria,Wind direction; hourly mean,Wind,Vent,Vento,Wind,H,0,Integer,°
4,erefaoh0,Referenzverdunstung nach FAO; Stundensumme,Évaporation de réference de FAO; somme horaire,Evaporazione referenze di FAO; somma oraria,Reference evaporation from FAO; hourly total,Verdunstung,Évaporation,Evaporazione,Evaporation,H,3,Float,mm/h
10,fkl010h0,Windgeschwindigkeit skalar; Stundenmittel in m/s,Vitesse du vent scalaire; moyenne horaire en m/s,Velocità del vento scalare; media oraria in m/s,Wind speed scalar; hourly mean in m/s,Wind,Vent,Vento,Wind,H,1,Float,m/s
11,fkl010h1,Böenspitze (Sekundenböe); Stundenmaximum in m/s,Rafale (intégration 1 s); maximum horaire en m/s,Raffica del vento (su un secondo); massima ora...,Gust peak (one second); hourly maximum in m/s,Wind,Vent,Vento,Wind,H,1,Float,m/s
12,fkl010h3,Böenspitze (3-Sekundenböe); Stundenmaximum in m/s,Rafale (intégration 3 s); maximum horaire en m/s,Raffica del vento (su tre secondi); massima or...,Gust peak (three seconds); hourly maximum in m/s,Wind,Vent,Vento,Wind,H,1,Float,m/s
23,fu3010h0,Windgeschwindigkeit skalar; Stundenmittel in km/h,Vitesse du vent scalaire; moyenne horaire en km/h,Velocità del vento scalare; media oraria in km/h,Wind speed scalar; hourly mean in km/h,Wind,Vent,Vento,Wind,H,1,Float,km/h
24,fu3010h1,Böenspitze (Sekundenböe); Stundenmaximum in km/h,Rafale (intégration 1 s); maximum horaire en km/h,Raffica del vento (su un secondo); massima ora...,Gust peak (one second); hourly maximum in km/h,Wind,Vent,Vento,Wind,H,1,Float,km/h
25,fu3010h3,Böenspitze (3-Sekundenböe); Stundenmaximum in ...,Rafale (intégration 3 s); maximum horaire en km/h,Raffica del vento (su tre secondi); massima or...,Gust peak (three seconds); hourly maximum in km/h,Wind,Vent,Vento,Wind,H,1,Float,km/h
33,fve010h0,Windgeschwindigkeit vektoriell; Stundenmittel ...,Vitesse du vent vectorielle; moyenne horaire e...,Velocità del vento vettoriale; media oraria in...,Wind speed vectorial; hourly mean in m/s,Wind,Vent,Vento,Wind,H,1,Float,m/s
36,gre000h0,Globalstrahlung; Stundenmittel,Rayonnement global; moyenne horaire,Radiazione globale; media oraria,Global radiation; hourly mean,Strahlung,Rayonnement,Radiazione,Radiation,H,0,Integer,W/m²


In [31]:
data_inventory

,station_abbr,parameter_shortname,meas_cat_nr,data_since,data_till,owner
0,ABO,dkl010d0,1,25.11.1983 00:00,NaN,MeteoSchweiz
1,ABO,dkl010h0,1,23.11.1983 10:00,NaN,MeteoSchweiz
2,ABO,dkl010z0,1,01.02.2004 00:00,NaN,MeteoSchweiz
3,ABO,erefaod0,1,25.11.1983 00:00,NaN,MeteoSchweiz
4,ABO,erefaoh0,1,01.05.2011 00:00,NaN,MeteoSchweiz
...,...,...,...,...,...,...
19520,ZER,xno000m0,1,01.01.1959 00:00,NaN,MeteoSchweiz
19521,ZER,xno000y0,1,01.01.1959 00:00,NaN,MeteoSchweiz
19522,ZER,xno012d0,1,03.12.1981 00:00,NaN,MeteoSchweiz
19523,ZER,xno012m0,1,01.12.1981 00:00,NaN,MeteoSchweiz


## hourly weather data
- use Zürich / Kloten as representative
- https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/klo/ogd-smn_klo_h_historical_2020-2029.csv

In [ ]:
weather_df = pd.read_csv(f"{EXT_DIR}/weather/ogd-smn_klo_h_historical_2020-2029.csv", encoding="latin-1", sep=";")
# weather_df = pd.read_csv(f"https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/klo/ogd-smn_klo_h_historical_2020-2029.csv", encoding="latin-1", sep=";")

In [33]:
weather_df

,station_abbr,reference_timestamp,tre200h0,tre200hn,tre200hx,tre005h0,tre005hn,ure200h0,pva200h0,tde200h0,prestah0,pp0qffh0,pp0qnhh0,ppz700h0,ppz850h0,fkl010h1,dkl010h0,fkl010h0,fu3010h0,fu3010h1,fkl010h3,fu3010h3,wcc006h0,fve010h0,rre150h0,htoauths,gre000h0,oli000h0,olo000h0,osr000h0,ods000h0,sre000h0,erefaoh0,tso005hs,tso010hs,tso020hs
0,KLO,01.01.2020 00:00,-1.6,-1.7,-1.4,-1.3,-1.4,96.7,5.3,-2.0,982.7,1036.6,1033.9,NaN,NaN,3.8,5.0,2.1,7.6,13.7,3.7,13.3,NaN,NaN,0.0,NaN,3,302,NaN,NaN,1.0,0,-0.002,2.6,3.3,4.1
1,KLO,01.01.2020 01:00,-1.6,-1.8,-1.4,-1.4,-1.6,96.8,5.3,-2.0,982.2,1036.1,1033.4,NaN,NaN,3.1,38.0,1.9,6.8,11.2,3.0,10.8,NaN,NaN,0.0,NaN,2,303,NaN,NaN,1.0,0,-0.002,2.5,3.3,4.1
2,KLO,01.01.2020 02:00,-1.6,-1.9,-1.4,-1.4,-1.6,96.9,5.3,-2.0,981.9,1035.8,1033.0,NaN,NaN,2.9,37.0,1.7,6.1,10.4,2.9,10.4,NaN,NaN,0.0,NaN,2,304,NaN,NaN,1.0,0,-0.003,2.5,3.2,4.1
3,KLO,01.01.2020 03:00,-1.5,-1.7,-1.3,-1.3,-1.4,97.2,5.3,-1.9,981.6,1035.4,1032.7,NaN,NaN,3.3,51.0,0.7,2.5,11.9,2.1,7.6,NaN,NaN,0.0,NaN,2,307,NaN,NaN,1.0,0,-0.004,2.5,3.2,4.1
4,KLO,01.01.2020 04:00,-1.3,-1.4,-1.2,-1.0,-1.1,97.3,5.4,-1.7,981.5,1035.3,1032.6,NaN,NaN,3.0,28.0,1.7,6.1,10.8,3.0,10.8,NaN,NaN,0.0,NaN,2,308,NaN,NaN,1.0,0,-0.003,2.5,3.2,4.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52603,KLO,31.12.2025 19:00,-4.3,-5.1,-3.4,-10.5,-11.5,64.9,2.9,-9.9,972.6,1026.5,1023.3,NaN,NaN,1.8,243.0,0.8,2.9,6.5,1.7,6.1,NaN,NaN,0.0,0.0,2,209,NaN,NaN,1.0,0,0.002,1.7,2.6,3.7
52604,KLO,31.12.2025 20:00,-5.6,-6.8,-4.1,-10.7,-11.3,72.6,2.9,-9.8,972.5,1026.6,1023.2,NaN,NaN,1.8,79.0,1.0,3.6,6.5,1.7,6.1,NaN,NaN,0.0,0.0,3,209,NaN,NaN,2.0,0,0.002,1.6,2.5,3.7
52605,KLO,31.12.2025 21:00,-6.7,-7.4,-5.7,-11.2,-11.5,76.7,2.9,-10.1,972.3,1026.7,1023.0,NaN,NaN,1.4,153.0,0.7,2.5,5.0,1.3,4.7,NaN,NaN,0.0,0.0,2,209,NaN,NaN,1.0,0,-0.001,1.5,2.5,3.7
52606,KLO,31.12.2025 22:00,-7.6,-8.6,-6.6,-11.5,-11.7,79.2,2.8,-10.5,972.1,1026.7,1022.8,NaN,NaN,1.7,306.0,0.9,3.2,6.1,1.6,5.8,NaN,NaN,0.0,0.0,2,209,NaN,NaN,1.0,0,0.000,1.4,2.4,3.6


In [13]:
weather_df.columns

Index(['station_abbr', 'reference_timestamp', 'tre200h0', 'tre200hn',
       'tre200hx', 'tre005h0', 'tre005hn', 'ure200h0', 'pva200h0', 'tde200h0',
       'prestah0', 'pp0qffh0', 'pp0qnhh0', 'ppz700h0', 'ppz850h0', 'fkl010h1',
       'dkl010h0', 'fkl010h0', 'fu3010h0', 'fu3010h1', 'fkl010h3', 'fu3010h3',
       'wcc006h0', 'fve010h0', 'rre150h0', 'htoauths', 'gre000h0', 'oli000h0',
       'olo000h0', 'osr000h0', 'ods000h0', 'sre000h0', 'erefaoh0', 'tso005hs',
       'tso010hs', 'tso020hs'],
      dtype='object')

In [16]:
parameters[((parameters['parameter_granularity'] == "H")&(parameters['parameter_shortname'].isin(weather_df.columns)))][['parameter_shortname', 'parameter_description_en', 'parameter_unit']]

,parameter_shortname,parameter_description_en,parameter_unit
1,dkl010h0,Wind direction; hourly mean,°
4,erefaoh0,Reference evaporation from FAO; hourly total,mm/h
10,fkl010h0,Wind speed scalar; hourly mean in m/s,m/s
11,fkl010h1,Gust peak (one second); hourly maximum in m/s,m/s
12,fkl010h3,Gust peak (three seconds); hourly maximum in m/s,m/s
23,fu3010h0,Wind speed scalar; hourly mean in km/h,km/h
24,fu3010h1,Gust peak (one second); hourly maximum in km/h,km/h
25,fu3010h3,Gust peak (three seconds); hourly maximum in km/h,km/h
33,fve010h0,Wind speed vectorial; hourly mean in m/s,m/s
36,gre000h0,Global radiation; hourly mean,W/m²


# COVID-19
## global
- https://github.com/owid/covid-19-data/blob/master/public/data/README.md

In [37]:
covid = pd.read_csv("https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv")

In [38]:
ch_covid = covid[(covid["iso_code"] == "CHE") & (covid["date"].between("2021-01-01", "2021-12-31"))]
ch_covid

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths_per_million,new_deaths_per_million,new_deaths_smoothed_per_million,reproduction_rate,icu_patients,icu_patients_per_million,hosp_patients,hosp_patients_per_million,weekly_icu_admissions,weekly_icu_admissions_per_million,weekly_hosp_admissions,weekly_hosp_admissions_per_million,total_tests,new_tests,total_tests_per_thousand,new_tests_per_thousand,new_tests_smoothed,new_tests_smoothed_per_thousand,positive_rate,tests_per_case,tests_units,total_vaccinations,people_vaccinated,people_fully_vaccinated,total_boosters,new_vaccinations,new_vaccinations_smoothed,total_vaccinations_per_hundred,people_vaccinated_per_hundred,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,new_vaccinations_smoothed_per_million,new_people_vaccinated_smoothed,new_people_vaccinated_smoothed_per_hundred,stringency_index,population_density,median_age,aged_65_older,aged_70_older,gdp_per_capita,extreme_poverty,cardiovasc_death_rate,diabetes_prevalence,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
371318,CHE,Europe,Switzerland,2021-01-01,435319.0,0.0,3397.86,7182.0,0.0,85.71,49512.07,0.00,386.46,816.86,0.00,9.75,0.99,421.0,48.17,2528.0,289.23,NaN,NaN,1088.0,124.48,3298473.0,10042.0,379.51,1.16,21188.0,2.44,0.16,6.1,tests performed,6393.0,6289.0,NaN,1.0,11.0,814.0,0.07,0.07,NaN,0.00,93.0,803.0,0.01,60.19,214.24,43.1,18.44,12.64,57410.17,NaN,99.74,5.59,22.6,28.9,NaN,4.53,83.78,0.96,8740471,NaN,NaN,NaN,NaN
371319,CHE,Europe,Switzerland,2021-01-02,435319.0,0.0,3397.86,7182.0,0.0,85.71,49512.07,0.00,386.46,816.86,0.00,9.75,1.00,421.0,48.17,2568.0,293.81,NaN,NaN,1124.0,128.60,3314419.0,15946.0,381.34,1.84,21292.0,2.45,0.17,6.0,tests performed,6397.0,6293.0,NaN,1.0,4.0,752.0,0.07,0.07,NaN,0.00,86.0,741.0,0.01,60.19,214.24,43.1,18.44,12.64,57410.17,NaN,99.74,5.59,22.6,28.9,NaN,4.53,83.78,0.96,8740471,NaN,NaN,NaN,NaN
371320,CHE,Europe,Switzerland,2021-01-03,458608.0,23289.0,3327.00,7756.0,574.0,82.00,52160.90,2648.83,378.40,882.15,65.28,9.33,1.01,428.0,48.97,2661.0,304.45,NaN,NaN,1141.0,130.54,3326777.0,12358.0,382.77,1.42,21229.0,2.44,0.17,6.0,tests performed,6402.0,6296.0,NaN,3.0,5.0,637.0,0.07,0.07,NaN,0.00,73.0,627.0,0.01,60.19,214.24,43.1,18.44,12.64,57410.17,NaN,99.74,5.59,22.6,28.9,NaN,4.53,83.78,0.96,8740471,8413.6,12.22,42.19,968.04
371321,CHE,Europe,Switzerland,2021-01-04,458608.0,0.0,3327.00,7756.0,0.0,82.00,52160.90,0.00,378.40,882.15,0.00,9.33,1.02,434.0,49.65,2697.0,308.56,NaN,NaN,1105.0,126.42,3356191.0,29414.0,386.15,3.38,21303.0,2.45,0.17,6.0,tests performed,10083.0,9930.0,NaN,4.0,3681.0,1046.0,0.12,0.11,NaN,0.00,120.0,1032.0,0.01,60.19,214.24,43.1,18.44,12.64,57410.17,NaN,99.74,5.59,22.6,28.9,NaN,4.53,83.78,0.96,8740471,NaN,NaN,NaN,NaN
371322,CHE,Europe,Switzerland,2021-01-05,458608.0,0.0,3327.00,7756.0,0.0,82.00,52160.90,0.00,378.40,882.15,0.00,9.33,1.01,443.0,50.68,2689.0,307.65,NaN,NaN,1088.0,124.48,3387522.0,31331.0,389.76,3.60,21571.0,2.48,0.16,6.1,tests performed,17224.0,17028.0,NaN,5.0,7141.0,1824.0,0.20,0.19,NaN,0.00,209.0,1809.0,0.02,60.19,214.24,43.1,18.44,12.64,57410.17,NaN,99.74,5.59,22.6,28.9,NaN,4.53,83.78,0.96,8740471,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371678,CHE,Europe,Switzerland,2021-12-27,1253630.0,0.0,9277.14,11828.0,0.0,21.71,142584.66,0.00,1055.16,1345.29,0.00,2.47,1.43,334.0,38.21,1721.0,196.90,NaN,NaN,827.0,94.62,14093106.0,58065.0,1621.50,6.68,57589.0,6.63,0.20,4.

## Swiss
documentation
- https://www.covid19.admin.ch/api/data/documentation
interesting things
- cases by region, age group
- Hospitalisations
- Virus variants

In [39]:
cases = pd.read_csv("https://covid19.admin.ch/api/data/20231206-0sxi4s4a/sources/COVID19Cases_geoRegion.csv", parse_dates=["datum"])
cases_ch = cases[cases["geoRegion"] == "CH"]

In [40]:
cases_ch

,geoRegion,datum,entries,sumTotal,timeframe_14d,timeframe_all,offset_last7d,sumTotal_last7d,offset_last14d,sumTotal_last14d,offset_last28d,sumTotal_last28d,sum7d,sum14d,mean7d,mean14d,entries_diff_last_age,pop,inz_entries,inzsumTotal,inzmean7d,inzmean14d,inzsumTotal_last7d,inzsumTotal_last14d,inzsumTotal_last28d,inzsum7d,inzsum14d,sumdelta7d,inzdelta7d,type,type_variant,version,datum_unit,entries_letzter_stand,entries_neu_gemeldet,entries_diff_last
0,CH,2020-02-24,1,1,False,True,4385008,0,4383801,0,4376250,0,NaN,NaN,NaN,NaN,7,8738791,0.01,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Cases,NaN,2023-01-24_06-03-16,day,1,0,914
1,CH,2020-02-25,1,2,False,True,4385008,0,4383801,0,4376250,0,NaN,NaN,NaN,NaN,7,8738791,0.01,0.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Cases,NaN,2023-01-24_06-03-16,day,1,0,914
2,CH,2020-02-26,10,12,False,True,4385008,0,4383801,0,4376250,0,NaN,NaN,NaN,NaN,7,8738791,0.11,0.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Cases,NaN,2023-01-24_06-03-16,day,10,0,914
3,CH,2020-02-27,10,22,False,True,4385008,0,4383801,0,4376250,0,NaN,NaN,8.14,NaN,7,8738791,0.11,0.25,0.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Cases,NaN,2023-01-24_06-03-16,day,10,0,914
4,CH,2020-02-28,10,32,False,True,4385008,0,4383801,0,4376250,0,NaN,NaN,12.29,NaN,7,8738791,0.11,0.37,0.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Cases,NaN,2023-01-24_06-03-16,day,10,0,914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,CH,2022-12-28,1052,4378584,False,True,4385008,0,4383801,0,4376250,2334,7785.0,20313.0,849.71,838.43,7,8738791,12.04,50105.15,9.72,9.59,NaN,NaN,26.71,89.09,232.45,-949.0,-10.86,COVID19Cases,NaN,2023-01-24_06-03-16,day,1049,3,914
1039,CH,2022-12-29,1001,4379585,False,True,4385008,0,4383801,0,4376250,3335,6990.0,19170.0,812.14,733.57,7,8738791,11.45,50116.60,9.29,8.39,NaN,NaN,38.16,79.99,219.37,-795.0,-9.10,COVID19Cases,NaN,2023-01-24_06-03-16,day,998,3,914
1040,CH,2022-12-30,878,4380463,False,True,4385008,0,4383801,0,4376250,4213,6253.0,18135.0,769.29,636.50,7,8738791,10.05,50126.65,8.80,7.28,NaN,NaN,48.21,71.55,207.52,-737.0,-8.44,COVID19Cases,NaN,2023-01-24_06-03-16,day,877,1,914
1041,CH,2022-12-31,570,4381033,False,True,4385008,0,4383801,0,4376250,4783,5948.0,17673.0,654.29,587.64,7,8738791,6.52,50133.17,7.49,6.72,NaN,NaN,54.73,68.06,202.24,-305.0,-3.49,COVID19Cases,NaN,2023-01-24_06-03-16,day,565,5,914


In [41]:
hosp = pd.read_csv("https://covid19.admin.ch/api/data/20231206-0sxi4s4a/sources/COVID19Hosp_geoRegion.csv", parse_dates=["datum"])
hosp

,geoRegion,datum,entries,sumTotal,timeframe_14d,timeframe_all,offset_last7d,sumTotal_last7d,offset_last14d,sumTotal_last14d,offset_last28d,sumTotal_last28d,sum7d,sum14d,mean7d,mean14d,entries_diff_last_age,pop,inz_entries,inzsumTotal,inzmean7d,inzmean14d,inzsumTotal_last7d,inzsumTotal_last14d,inzsumTotal_last28d,inzsum7d,inzsum14d,sumdelta7d,inzdelta7d,type,type_variant,version,datum_unit,entries_letzter_stand,entries_neu_gemeldet,entries_diff_last
0,CH,2020-02-24,5,5,False,True,63897,0,63824,0,63502,0,NaN,NaN,NaN,NaN,7,8738791,0.06,0.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Hosp,NaN,2023-01-24_06-03-16,day,5,0,86
1,CH,2020-02-25,4,9,False,True,63897,0,63824,0,63502,0,NaN,NaN,NaN,NaN,7,8738791,0.05,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Hosp,NaN,2023-01-24_06-03-16,day,4,0,86
2,CH,2020-02-26,9,18,False,True,63897,0,63824,0,63502,0,NaN,NaN,NaN,NaN,7,8738791,0.10,0.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Hosp,NaN,2023-01-24_06-03-16,day,9,0,86
3,CH,2020-02-27,5,23,False,True,63897,0,63824,0,63502,0,NaN,NaN,7.43,NaN,7,8738791,0.06,0.26,0.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Hosp,NaN,2023-01-24_06-03-16,day,5,0,86
4,CH,2020-02-28,5,28,False,True,63897,0,63824,0,63502,0,NaN,NaN,8.57,NaN,7,8738791,0.06,0.32,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COVID19Hosp,NaN,2023-01-24_06-03-16,day,5,0,86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30242,ZH,2022-12-28,1,9129,False,True,9161,0,9154,0,9125,4,25.0,66.0,2.86,2.79,7,1564662,0.06,583.45,0.18,0.18,NaN,NaN,0.26,1.60,4.22,-3.0,-0.19,COVID19Hosp,NaN,2023-01-24_06-03-16,day,1,0,4
30243,ZH,2022-12-29,3,9132,False,True,9161,0,9154,0,9125,7,21.0,63.0,2.29,2.43,7,1564662,0.19,583.64,0.15,0.16,NaN,NaN,0.45,1.34,4.03,-4.0,-0.26,COVID19Hosp,NaN,2023-01-24_06-03-16,day,3,0,4
30244,ZH,2022-12-30,3,9135,False,True,9161,0,9154,0,9125,10,20.0,62.0,1.86,2.29,7,1564662,0.19,583.83,0.12,0.15,NaN,NaN,0.64,1.28,3.96,-1.0,-0.06,COVID19Hosp,NaN,2023-01-24_06-03-16,day,2,1,4
30245,ZH,2022-12-31,0,9135,False,True,9161,0,9154,0,9125,10,20.0,58.0,2.00,2.50,7,1564662,0.00,583.83,0.13,0.16,NaN,NaN,0.64,1.28,3.71,0.0,0.00,COVID19Hosp,NaN,2023-01-24_06-03-16,day,0,0,4


In [42]:
hosp['geoRegion'].unique()

array(['CH', 'CHFL', 'AG', 'AI', 'AR', 'BE', 'BL', 'BS', 'FL', 'FR', 'GE',
       'GL', 'GR', 'JU', 'LU', 'NE', 'NW', 'OW', 'SG', 'SH', 'SO', 'SZ',
       'TG', 'TI', 'UR', 'VD', 'VS', 'ZG', 'ZH'], dtype=object)

In [ ]:
# alpha to delta shift in mid 2021
variants = pd.read_csv("https://covid19.admin.ch/api/data/20231206-0sxi4s4a/sources/COVID19Variants_wgs.csv")
variants

,geoRegion,variant_type,date,entries,sumTotal,freq,prct_mean7d,prct,prct_lower_ci,prct_upper_ci,timeframe_all,type,version,classification,data_source
0,CHFL,all_sequenced,2020-09-28,4,4,NaN,NaN,NaN,NaN,NaN,True,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
1,CHFL,B.1.1.318,2020-09-28,0,0,NaN,NaN,NaN,NaN,NaN,True,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
2,CHFL,B.1.1.529,2020-09-28,0,0,NaN,NaN,NaN,NaN,NaN,True,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
3,CHFL,B.1.1.7,2020-09-28,0,0,NaN,NaN,NaN,NaN,NaN,True,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
4,CHFL,B.1.351,2020-09-28,0,0,NaN,NaN,NaN,NaN,NaN,True,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9688,CHFL,B.1.617.2,2022-10-07,0,46771,0.0,NaN,0.0,0.0,56.7,False,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
9689,CHFL,C.37,2022-10-07,0,29,0.0,NaN,0.0,0.0,56.7,False,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
9690,CHFL,other_lineages,2022-10-07,0,18293,0.0,NaN,0.0,0.0,56.7,False,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs
9691,CHFL,P.1,2022-10-07,0,231,0.0,NaN,0.0,0.0,56.7,False,COVID19VirusVariants,2022-10-18_10-53-01,default,wgs


# Holidays

In [ ]:
import pandas as pd
import holidays

# --- Public holidays (Zurich canton) ---
zh_holidays = holidays.country_holidays("CH", subdiv="ZH", years=2021)
public_hols = pd.to_datetime(list(zh_holidays.keys()))

# --- School holidays (Zurich 2021, from schulferien.org) ---
school_breaks = [
    ("2021-04-26", "2021-05-08"),
    ("2021-07-19", "2021-08-21"),
    ("2021-10-11", "2021-10-23"),
    ("2021-12-20", "2022-01-01"),
]

# --- Build daily calendar ---
calendar = pd.DataFrame({"date": pd.date_range("2021-01-01", "2021-12-31", freq="D")})
calendar["is_weekend"]    = calendar["date"].dt.dayofweek >= 5
calendar["is_public_hol"] = calendar["date"].isin(public_hols)
calendar["is_school_break"] = False
for start, end in school_breaks:
    calendar.loc[calendar["date"].between(start, end), "is_school_break"] = True

calendar["official_day_off"] = calendar["is_weekend"] | calendar["is_public_hol"] | calendar["is_school_break"]

# --- Bridge days (workday sandwiched between two days off) ---
prev_off = calendar["official_day_off"].shift(1, fill_value=False)
next_off = calendar["official_day_off"].shift(-1, fill_value=False)
calendar["is_bridge_day"] = ~calendar["official_day_off"] & ~calendar["is_weekend"] & prev_off & next_off

# --- School shoulder days (1 day before/after a school break) ---
calendar["near_school_break"] = (
    calendar["is_school_break"].shift(-1, fill_value=False)
    | calendar["is_school_break"].shift(1, fill_value=False)
) & ~calendar["is_school_break"] & ~calendar["is_weekend"]

# --- Day type label ---
def day_type(row):
    if row["is_public_hol"]:      return "public_holiday"
    if row["is_bridge_day"]:      return "bridge_day"
    if row["is_weekend"]:         return "weekend"
    return "workday"

calendar["day_type"] = calendar.apply(day_type, axis=1)
calendar["is_extended_day_off"] = calendar["day_type"] != "workday"

calendar.to_csv("../data/external/holidays/zurich_calendar_2021.csv", index=False)
calendar["day_type"].value_counts()

day_type
workday           255
weekend            97
public_holiday     12
bridge_day          1
Name: count, dtype: int64

In [52]:
calendar

,date,is_weekend,is_public_hol,is_school_break,official_day_off,is_bridge_day,near_school_break,day_type,is_extended_day_off
0,2021-01-01,False,True,False,True,False,False,public_holiday,True
1,2021-01-02,True,True,False,True,False,False,public_holiday,True
2,2021-01-03,True,False,False,True,False,False,weekend,True
3,2021-01-04,False,False,False,False,False,False,workday,False
4,2021-01-05,False,False,False,False,False,False,workday,False
...,...,...,...,...,...,...,...,...,...
360,2021-12-27,False,False,True,True,False,False,workday,False
361,2021-12-28,False,False,True,True,False,False,workday,False
362,2021-12-29,False,False,True,True,False,False,workday,False
363,2021-12-30,False,False,True,True,False,False,workday,False
